# MedGemma 1.5 - Chest X-ray Analysis (Colab GPU Demo)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DrFarooqAi/medgemma-xray-demo/blob/main/medgemma_colab_demo.ipynb)

Run Google's **MedGemma 1.5 4B** on a free Colab **T4 GPU** and get a streaming AI radiology report from a chest X-ray in seconds.

This mirrors the Hugging Face Space ([farooqgenai/medgemma-xray-demo](https://huggingface.co/spaces/farooqgenai/medgemma-xray-demo)) but on a GPU, so inference is fast.

**Before you run:**
1. `Runtime -> Change runtime type -> T4 GPU -> Save`
2. Accept the model license (one click) at https://huggingface.co/google/medgemma-1.5-4b-it
3. Have a Hugging Face token ready: https://huggingface.co/settings/tokens

> **Not for clinical use.** Educational and research purposes only. Built by Dr. Muhammad Farooq.

### 1. Install dependencies
If you later hit a Pillow error, use `Runtime -> Restart session` and re-run from step 2 (skip this cell).

In [ ]:
!pip install -q -U transformers accelerate

### 2. Log in with your Hugging Face token

In [ ]:
from huggingface_hub import login
from getpass import getpass
login(token=getpass('Paste your HF token (hf_...): '))

### 3. Load MedGemma 1.5 4B on the GPU
Picks `bfloat16` on modern GPUs, `float16` on T4 (Turing lacks bf16).

In [ ]:
import torch, time
from transformers import pipeline

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0), '| dtype:', dtype)

t = time.time()
pipe = pipeline(
    'image-text-to-text',
    model='google/medgemma-1.5-4b-it',
    dtype=dtype,
    device_map='auto',
    low_cpu_mem_usage=True,
)
print(f'Model loaded in {time.time()-t:.0f}s on {pipe.model.device}')

### 4. Upload a chest X-ray

In [ ]:
from google.colab import files
from PIL import Image

up = files.upload()                       # pick a chest X-ray (PNG/JPG)
img_path = next(iter(up))
image = Image.open(img_path).convert('RGB')
image

### 5. Generate the report (streaming)
Uses `pipe.processor` (not `pipe.tokenizer`) so the model actually receives the image via `pixel_values`.

In [ ]:
from threading import Thread
from transformers import TextIteratorStreamer

model = pipe.model
processor = pipe.processor

messages = [{'role': 'user', 'content': [
    {'type': 'image', 'image': image},
    {'type': 'text', 'text': 'Describe this chest X-ray.'},
]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors='pt',
).to(model.device)
print('pixel_values present:', 'pixel_values' in inputs, '| input tokens:', inputs['input_ids'].shape[-1])

streamer = TextIteratorStreamer(pipe.tokenizer, skip_prompt=True, skip_special_tokens=True)
Thread(target=model.generate, kwargs=dict(
    **inputs, max_new_tokens=300, streamer=streamer, do_sample=False)).start()

t = time.time()
print('\n===== AI RADIOLOGY REPORT =====\n')
for tok in streamer:
    print(tok, end='', flush=True)
print(f'\n\n[done in {time.time()-t:.0f}s]')